In [1]:
import re
import unicodedata

def normalize(text):

    text = unicodedata.normalize("NFKC", text)


    text = re.sub(r"[‘’ʻʼ`´]", "'", text)


    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)


    text = text.lower()


    text = re.sub(r"[^\w\s']", " ", text)


    text = re.sub(r"\s+", " ", text)

    return text.strip()


with open("../../data/test/test_book.txt", encoding="utf-8") as f:
    book = f.read()

with open("../../data/test/transcription.txt", encoding="utf-8") as f:
    transcript = f.read()

book_clean = normalize(book)
transcript_clean = normalize(transcript)

In [2]:
print("BOOK:")
print(book_clean[:1000])

print("\nTRANSCRIPTION:")
print(transcript_clean[:1000])

print("\nWORD COUNTS:")
print("Book:", len(book_clean.split()))
print("Transcription:", len(transcript_clean.split()))

BOOK:
ain ochilar y adim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirarkan kunlarning birida o'ng 401 vaziri undan podshohim nimagadir mudom фатат 512 boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q o'g'il ko'rsam boshqa armonim qolmaydi debdi podsho olampanoh sizga maslahatim bor chinmochin mamlakatida bir daraxt bor u daraxti baxt deb ataladi uning bargini ming dardga shifo deydilar shu bargdan olib kelishni buyuring podsho vaziridan bu maslahatni eshitib xursand bo'libdi va shaharga jarchi qo'ydiribdi kimki chinmochin mamlakatiga borib podsho uchun daraxti baxtning bargidan olib kelsa ul bahodirga xazinaning yarmisi in'om etiladi bu ovozani eshitib shoyad olib kelolsak oliy hazratning in'omiga muyassar bo'lardik deydiganlar ko'payib qolibdiyu lekin birontasi ham yurak yutib podsho huzuriga kelmabdi buxorodan ancha uzoq bir qishloqdan baxtiyor degan cho'pon yigit shoh saroyiga kelib jarchilaringizdan eshitdimki c

In [3]:
from pathlib import Path

transcription_path = Path("../../data/test/transcription.txt")
transcription_path.parent.mkdir(parents=True, exist_ok=True)
transcription_path.write_text(transcript_clean, encoding="utf-8")

6304

In [4]:
book_path = Path("../../data/test/book.txt")
book_path.parent.mkdir(parents=True, exist_ok=True)
book_path.write_text(book_clean, encoding="utf-8")

6457

In [5]:
print("TRANSCRIPTION:")
print(transcript_clean[:500])

print("\nBOOK:")
print(book_clean[:500])

TRANSCRIPTION:
qadrli bolajonlar o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman bolalar adabiyoti jurnali qadim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirar ekan kunlarning birida o'ng qo'li vaziri undan podshohim nimagadir mudom g'amginsiz boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q o'g'il ko'rsam boshqa armonim qolmaydi debdi podsho olampanoh sizga maslahatim bor chinmochin chinmochin mamlakatiga borib pod

BOOK:
ain ochilar y adim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirarkan kunlarning birida o'ng 401 vaziri undan podshohim nimagadir mudom фатат 512 boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q o'g'il ko'rsam boshqa armonim qolmaydi debdi podsho olampanoh sizga maslahatim bor chinmochin mamlakatida bir daraxt bor u daraxti baxt deb ataladi uning bargini ming dardga shifo deydilar shu bargdan olib kelish

In [6]:
from rapidfuzz import fuzz

book_words = book_clean.split()
asr_words = transcript_clean.split()

def find_match(start, window=40):
    query = " ".join(asr_words[start:start + window])

    best_score = 0
    best_position = 0

    # Search the book in 5-word steps
    for i in range(0, len(book_words) - window + 1, 5):
        candidate = " ".join(book_words[i:i + window])

        score = fuzz.ratio(query, candidate)

        if score > best_score:
            best_score = score
            best_position = i

    # Refine the result around the best position
    for i in range(max(0, best_position - 5),
                   min(len(book_words) - window + 1, best_position + 6)):
        candidate = " ".join(book_words[i:i + window])
        score = fuzz.ratio(query, candidate)

        if score > best_score:
            best_score = score
            best_position = i

    print(f"\nTRANSCRIPTION (starting at word {start}):")
    print(query)

    print("\nMATCHED BOOK TEXT:")
    print(" ".join(book_words[best_position:best_position + window]))

    print(f"\nSimilarity: {best_score:.1f}%")
    print(f"Book position: {best_position}")

    return best_position, best_score


# Search three different sections
match1 = find_match(0)
match2 = find_match(300)
match3 = find_match(600)


TRANSCRIPTION (starting at word 0):
qadrli bolajonlar o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman bolalar adabiyoti jurnali qadim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirar ekan kunlarning birida o'ng qo'li vaziri undan podshohim nimagadir mudom g'amginsiz boisi nima deb

MATCHED BOOK TEXT:
ain ochilar y adim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirarkan kunlarning birida o'ng 401 vaziri undan podshohim nimagadir mudom фатат 512 boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim

Similarity: 58.8%
Book position: 0

TRANSCRIPTION (starting at word 300):
oting nima kimning jigargo'shasisan deb so'rabdi men xuroson podshosining qizi bo'laman ismim gulnora devlar meni bu yerga o'g'irlab olib kelgan bu yer devlarning makoni hali zamon devlar kelib qoladi o'zingni nega xarob qilmoqchisan tezroq bu yerdan jo'nab qolgin debdi qiz

MATCHED BOOK TEXT:
oting nima kimning jigargo''sh

In [7]:
print("Match 1:", match1)
print("Match 2:", match2)
print("Match 3:", match3)

Match 1: (0, 58.78003696857671)
Match 2: (324, 99.81785063752277)
Match 3: (629, 100.0)


In [8]:
from difflib import SequenceMatcher

# Find the approximate corresponding region in the book
start = match1[0]
end = match3[0] + (len(asr_words) - 600)

book_section = book_words[start:end]

# Compare the two word sequences
matcher = SequenceMatcher(
    None,
    book_section,
    asr_words,
    autojunk=False
)

# Show differences
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        continue

    print("\nDIFFERENCE:", tag)
    print("BOOK:", " ".join(book_section[i1:i2])[:250])
    print("ASR: ", " ".join(asr_words[j1:j2])[:250])


DIFFERENCE: replace
BOOK: ain ochilar y adim
ASR:  qadrli bolajonlar o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman bolalar adabiyoti jurnali qadim

DIFFERENCE: replace
BOOK: o'tirarkan
ASR:  o'tirar ekan

DIFFERENCE: replace
BOOK: 401
ASR:  qo'li

DIFFERENCE: replace
BOOK: фатат 512
ASR:  g'amginsiz

DIFFERENCE: delete
BOOK: mamlakatida bir daraxt bor u daraxti baxt deb ataladi uning bargini ming dardga shifo deydilar shu bargdan olib kelishni buyuring podsho vaziridan bu maslahatni eshitib xursand bo'libdi va shaharga jarchi qo'ydiribdi kimki
ASR:  

DIFFERENCE: replace
BOOK: ko'payib qolibdiyu
ASR:  ko 'payib qolibdi yu

DIFFERENCE: replace
BOOK: birontasi
ASR:  birortasi

DIFFERENCE: replace
BOOK: podsho
ASR:  podshoh

DIFFERENCE: insert
BOOK: 
ASR:  podsho

DIFFERENCE: replace
BOOK: amaldorlariga
ASR:  amaldorlarga

DIFFERENCE: delete
BOOK: o'zbek xalq ertaklari т 1979
ASR:  

DIFFERENCE: replace
BOOK: kambag''allarning
ASR:  kambag'allarning

DIFFERENCE